In [1]:
!pip install -q segmentation-models-pytorch rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import rasterio
from tqdm import tqdm
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

DATA_DIR = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"

Device: cuda


In [3]:
def load_split(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_ids = load_split(f"{DATA_DIR}/split/train.txt")
val_ids   = load_split(f"{DATA_DIR}/split/val.txt")

all_ids = train_ids + val_ids

test_ids = [f.replace("_image.tif","")
            for f in os.listdir(f"{DATA_DIR}/prediction/image")]

print(len(all_ids), len(test_ids))

69 19


In [4]:
def preprocess(img):
    hh, hv = img[0], img[1]
    green, red, nir, swir = img[2], img[3], img[4], img[5]

    eps = 1e-6

    ndwi  = (green - nir) / (green + nir + eps)
    mndwi = (green - swir) / (green + swir + eps)
    ndvi  = (nir - red) / (nir + red + eps)
    sar_diff = hh - hv

    bands = [hh, hv, green, red, nir, swir, ndwi, mndwi, ndvi, sar_diff]

    normed = []
    for b in bands:
        p2, p98 = np.percentile(b, [2,98])
        b = np.clip(b, p2, p98)
        b = (b - p2)/(p98 - p2 + eps)
        normed.append(b)

    return np.stack(normed).astype(np.float32)

In [5]:
# =========================
# FIXED DATASET (WITH PSEUDO SUPPORT)
# =========================

class FloodDataset(Dataset):
    def __init__(self, ids, mode="train"):
        self.ids = ids
        self.mode = mode

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]

        # 🔥 Load correct image path
        if pid in test_ids:
            img_path = f"{DATA_DIR}/prediction/image/{pid}_image.tif"
        else:
            img_path = f"{DATA_DIR}/image/{pid}_image.tif"

        with rasterio.open(img_path) as src:
            img = src.read().astype(np.float32)

        img = preprocess(img)
        img = torch.from_numpy(img)

        # 🔥 CHECK FOR PSEUDO LABEL FIRST
        pseudo_path = f"/kaggle/working/pseudo_labels/{pid}_pseudo.npy"

        if os.path.exists(pseudo_path):
            mask = np.load(pseudo_path)
        else:
            if self.mode != "test":
                with rasterio.open(f"{DATA_DIR}/label/{pid}_label.tif") as src:
                    mask = src.read(1)
            else:
                return img, pid

        # 🔥 IMPORTANT FIX
        # =========================
        # FINAL SAFE MASK FIX
        # =========================
        
        mask = mask.astype(np.int64)
        
        # 🔥 FORCE VALID RANGE
        mask[mask < 0] = 255
        mask[mask > 2] = 255
        
        return img, torch.from_numpy(mask).long()

In [6]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=10,
    classes=3,
    decoder_attention_type="scse"
).to(device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [7]:
# =========================
# FIXED LOSS FUNCTION (PSEUDO LABEL SAFE)
# =========================

weights = torch.tensor([0.3, 5.0, 1.0]).to(device)

# 🔥 IMPORTANT FIX: ignore 255 pixels
ce = nn.CrossEntropyLoss(weight=weights, ignore_index=255)

dice = smp.losses.DiceLoss(mode="multiclass", ignore_index=255)

def loss_fn(logits, targets):
    return 0.4 * ce(logits, targets) + 0.6 * dice(logits, targets)

In [8]:
train_loader = DataLoader(FloodDataset(all_ids,"train"), batch_size=4, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(20):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(imgs), masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss: {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "/kaggle/working/model2.pth")

100%|██████████| 18/18 [00:18<00:00,  1.02s/it]


Epoch 0 Loss: 0.8754


100%|██████████| 18/18 [00:11<00:00,  1.55it/s]


Epoch 1 Loss: 0.7438


100%|██████████| 18/18 [00:11<00:00,  1.54it/s]


...

Epoch 17 Loss: 0.3975


100%|██████████| 18/18 [00:12<00:00,  1.48it/s]


Epoch 18 Loss: 0.3866


100%|██████████| 18/18 [00:12<00:00,  1.47it/s]


Epoch 19 Loss: 0.3877


In [9]:
# =========================
# TRAIN FRIEND MODEL (FAST REBUILD)
# =========================

import segmentation_models_pytorch as smp

model1 = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b5",
    encoder_weights="imagenet",
    in_channels=10,
    classes=3,
    decoder_attention_type="scse"
).to(device)

weights = torch.tensor([0.3, 5.0, 1.0]).to(device)
ce = nn.CrossEntropyLoss(weight=weights)
dice = smp.losses.DiceLoss(mode="multiclass")

def loss_fn(logits, targets):
    return 0.4 * ce(logits, targets) + 0.6 * dice(logits, targets)

train_loader = DataLoader(FloodDataset(all_ids,"train"), batch_size=4, shuffle=True)

optimizer = torch.optim.AdamW(model1.parameters(), lr=2e-4)

# 🔥 ONLY TRAIN 10 EPOCHS (ENOUGH FOR ENSEMBLE)
for epoch in range(10):
    model1.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        loss = loss_fn(model1(imgs), masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Model1] Epoch {epoch} Loss: {total_loss/len(train_loader):.4f}")

torch.save(model1.state_dict(), "/kaggle/working/best.pth")

print("✅ Friend model saved as best.pth")

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

100%|██████████| 18/18 [00:24<00:00,  1.36s/it]


[Model1] Epoch 0 Loss: 0.7963


100%|██████████| 18/18 [00:23<00:00,  1.33s/it]


...

[Model1] Epoch 7 Loss: 0.4138


100%|██████████| 18/18 [00:23<00:00,  1.33s/it]


[Model1] Epoch 8 Loss: 0.4199


100%|██████████| 18/18 [00:23<00:00,  1.33s/it]


[Model1] Epoch 9 Loss: 0.4162
✅ Friend model saved as best.pth


In [10]:
# =========================
# SAVE MODEL2 PROBS
# =========================

model.load_state_dict(torch.load("/kaggle/working/model2.pth"))
model.eval()

for pid in test_ids:
    with rasterio.open(f"{DATA_DIR}/prediction/image/{pid}_image.tif") as src:
        img = preprocess(src.read().astype(np.float32))

    img = torch.from_numpy(img).unsqueeze(0).float().to(device)

    with torch.no_grad():
        probs = torch.softmax(model(img), dim=1)[0].cpu().numpy()

    np.save(f"/kaggle/working/{pid}_model2.npy", probs)

print("✅ Model2 probs saved")

✅ Model2 probs saved


In [11]:
# =========================
# FINAL ENSEMBLE
# =========================

model1.load_state_dict(torch.load("/kaggle/working/best.pth"))
model1.eval()

def rle(mask):
    pixels=(mask==1).astype(np.uint8).flatten(order="F")
    pixels=np.concatenate([[0],pixels,[0]])
    runs=np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2]-=runs[::2]
    return " ".join(map(str,runs)) if len(runs)>0 else "0 0"

rows = []

for pid in tqdm(test_ids):

    with rasterio.open(f"{DATA_DIR}/prediction/image/{pid}_image.tif") as src:
        img = preprocess(src.read().astype(np.float32))

    img_tensor = torch.from_numpy(img).unsqueeze(0).float().to(device)

    with torch.no_grad():
        probs1 = torch.softmax(model1(img_tensor), dim=1)[0].cpu().numpy()

    probs2 = np.load(f"/kaggle/working/{pid}_model2.npy")

    # 🔥 ENSEMBLE
    final_probs = 0.8 * probs1 + 0.2 * probs2
    

    pred = final_probs.argmax(0)

    rows.append({"id": pid, "rle_mask": rle(pred)})

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/submission_ensemblev4.csv", index=False)

print("🔥 FINAL SUBMISSION READY")

100%|██████████| 19/19 [00:03<00:00,  6.10it/s]

🔥 FINAL SUBMISSION READY


In [12]:
# =========================
# PSEUDO LABEL GENERATION (CLEAN VERSION)
# =========================

import os
import numpy as np
import torch
import rasterio
from tqdm import tqdm

# 🔥 Create folder
pseudo_dir = "/kaggle/working/pseudo_labels"
os.makedirs(pseudo_dir, exist_ok=True)

pseudo_ids = []

for pid in tqdm(test_ids):

    # ---------- Load image ----------
    with rasterio.open(f"{DATA_DIR}/prediction/image/{pid}_image.tif") as src:
        img = preprocess(src.read().astype(np.float32))

    img_tensor = torch.from_numpy(img).unsqueeze(0).float().to(device)

    # ---------- Model 1 ----------
    with torch.no_grad():
        probs1 = torch.softmax(model1(img_tensor), dim=1)[0].cpu().numpy()

    # ---------- Model 2 ----------
    probs2 = np.load(f"/kaggle/working/{pid}_model2.npy")

    # ---------- Ensemble ----------
    final_probs = 0.8 * probs1 + 0.2 * probs2

    # ---------- Confidence ----------
    confidence = final_probs.max(0)
    pred = final_probs.argmax(0)

    # 🔥 Keep only HIGH confidence pixels
    pseudo_mask = np.where(confidence > 0.9, pred, 255).astype(np.uint8)

    # ---------- Save ----------
    save_path = f"{pseudo_dir}/{pid}_pseudo.npy"
    np.save(save_path, pseudo_mask)

    pseudo_ids.append(pid)

print("✅ Pseudo-labels saved:", len(pseudo_ids))

100%|██████████| 19/19 [00:02<00:00,  6.37it/s]

✅ Pseudo-labels saved: 19


In [13]:
# =========================
# DATASET WITH PSEUDO LABEL SUPPORT
# =========================

class FloodDatasetPseudo(Dataset):
    def __init__(self, ids, mode="train"):
        self.ids = ids
        self.mode = mode

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]

        if pid in test_ids:
            img_path = f"{DATA_DIR}/prediction/image/{pid}_image.tif"
        else:
            img_path = f"{DATA_DIR}/image/{pid}_image.tif"

        with rasterio.open(img_path) as src:
            img = preprocess(src.read().astype(np.float32))

        img = torch.from_numpy(img)

        # 🔥 Use pseudo label if exists
        pseudo_path = f"/kaggle/working/pseudo_labels/{pid}_pseudo.npy"

        if os.path.exists(pseudo_path):
            mask = np.load(pseudo_path)
        else:
            with rasterio.open(f"{DATA_DIR}/label/{pid}_label.tif") as src:
                mask = src.read(1)

        return img, torch.from_numpy(mask).long()

In [14]:
# =========================
# FINAL SAFE TRAINING CELL (NO MORE CRASH)
# =========================

model_final = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b5",
    encoder_weights="imagenet",
    in_channels=10,
    classes=3,
    decoder_attention_type="scse"
).to(device)

optimizer = torch.optim.AdamW(model_final.parameters(), lr=1e-4)

for epoch in range(10):
    model_final.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):

        imgs = imgs.to(device)

        # 🔥 FORCE SAFE MASK (CRITICAL FIX)
        masks = masks.numpy()
        masks[(masks < 0) | (masks > 2)] = 255
        masks = torch.from_numpy(masks).long().to(device)

        # 🔥 EXTRA SAFETY (NO FLOAT ISSUE)
        masks = masks.long()

        # DEBUG (runs once)
        if epoch == 0:
            print("Mask values:", torch.unique(masks))

        optimizer.zero_grad()

        outputs = model_final(imgs)

        # 🔥 FINAL SAFETY CHECK
        if outputs.shape[1] != 3:
            print("ERROR: wrong output channels:", outputs.shape)
            break

        loss = loss_fn(outputs, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Pseudo Train] Epoch {epoch} Loss: {total_loss/len(train_loader):.4f}")

torch.save(model_final.state_dict(), "/kaggle/working/final_pseudo.pth")

print("✅ TRAINING COMPLETED WITHOUT CRASH")

  0%|          | 0/18 [00:00<?, ?it/s]

Mask values: tensor([0, 1, 2], device='cuda:0')


  6%|▌         | 1/18 [00:01<00:24,  1.42s/it]

Mask values: tensor([0, 1, 2], device='cuda:0')


 11%|█         | 2/18 [00:02<00:22,  1.40s/it]

...

[Pseudo Train] Epoch 7 Loss: 0.4707


100%|██████████| 18/18 [00:23<00:00,  1.33s/it]


[Pseudo Train] Epoch 8 Loss: 0.4527


100%|██████████| 18/18 [00:23<00:00,  1.33s/it]


[Pseudo Train] Epoch 9 Loss: 0.4551
✅ TRAINING COMPLETED WITHOUT CRASH


In [15]:
# =========================
# FINAL INFERENCE + SUBMISSION (PSEUDO MODEL)
# =========================

import pandas as pd
from tqdm import tqdm

# 🔥 LOAD MODEL
model_final.load_state_dict(torch.load("/kaggle/working/final_pseudo.pth"))
model_final.eval()

# 🔥 RLE FUNCTION
def rle(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs)) if len(runs) else "0 0"


rows = []

for pid in tqdm(test_ids):

    # ---------- LOAD IMAGE ----------
    with rasterio.open(f"{DATA_DIR}/prediction/image/{pid}_image.tif") as src:
        img = preprocess(src.read().astype(np.float32))

    img = torch.from_numpy(img).unsqueeze(0).float().to(device)

    # ---------- PREDICT ----------
    with torch.no_grad():
        probs = torch.softmax(model_final(img), dim=1)[0].cpu().numpy()

    # 🔥 OPTIONAL FINAL BOOST (SAFE)
    probs[1] *= 1.1
    probs = probs / probs.sum(axis=0, keepdims=True)

    pred = probs.argmax(0)

    # 🔥 FLOOD MASK ONLY
    flood_mask = (pred == 1).astype(np.uint8)

    rows.append({
        "id": pid,
        "rle_mask": rle(flood_mask)
    })


# 🔥 SAVE CSV
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/submission_final_p.csv", index=False)

print("🔥 FINAL SUBMISSION GENERATED → submission_final_p.csv")
print("Flood predictions:", (df.rle_mask != "0 0").sum(), "/19")

100%|██████████| 19/19 [00:03<00:00,  6.26it/s]

🔥 FINAL SUBMISSION GENERATED → submission_final_p.csv
Flood predictions: 19 /19
